# Structural Variant ML Features - Exploratory Data Analysis
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  
**Table:** structural_variant_ml_features (216,951 SVs, 33 columns)

## Objective
Explore structural variant features: SV types, size distributions, gene overlap counts, breakpoint impacts, and clinical priority classifications.

## Key Questions
1. What is the distribution of SV types and size categories?
2. How many genes do SVs typically overlap?
3. What is the sv_classification distribution for target definition?
4. How does gene disruption fraction relate to clinical priority?
5. Which gene categories are most commonly affected?

## Deliverables
- 12+ visualizations saved to structural_variant_ml_features/images/
- EDA report saved to structural_variant_ml_features/reports/
- Missing values, correlation matrix, feature statistics saved to structural_variant_ml_features/metrics/

## Note
gene_list column is stored as pipe-delimited string (BRCA1|TP53|EGFR).
This notebook uses genes_overlapped (INT count) for analysis.
For ML training: gene_list will be dropped and genes_overlapped used instead.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path().absolute().parent.parent
BASE_OUT     = PROJECT_ROOT / 'data' / 'analytical' / 'structural_variant_ml_features'
IMAGES_DIR   = BASE_OUT / 'images'
REPORTS_DIR  = BASE_OUT / 'reports'
METRICS_DIR  = BASE_OUT / 'metrics'

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print(f"Images  : {IMAGES_DIR}")
print(f"Reports : {REPORTS_DIR}")
print(f"Metrics : {METRICS_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

## 3. Data Loading

In [ ]:
# structural_variant_ml_features has 216,951 rows - load full table
print("Loading structural_variant_ml_features (full table, 216K rows)...")
query = "SELECT * FROM gold.structural_variant_ml_features"
df = pd.read_sql(query, engine)

print(f"Rows loaded  : {len(df):,}")
print(f"Columns      : {len(df.columns)}")
print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 4. Type Conversion

In [ ]:
# INT columns from gold schema
int_cols = [
    'start_pos', 'end_pos', 'sv_size', 'genes_overlapped',
    'pharmacogenes_affected', 'omim_genes_affected',
    'kinase_genes_affected', 'receptor_genes_affected',
    'total_disease_associations', 'cancer_genes_affected',
    'neuro_genes_affected', 'broadly_expressed_genes_affected',
    'sv_combined_impact_score'
]

# DOUBLE columns from gold schema
double_cols = [
    'max_gene_disruption_fraction',
    'avg_druggability_affected_genes'
]

# BOOLEAN columns from gold schema
bool_cols = [
    'has_critical_gene_disruption',
    'has_disease_associated_genes',
    'affects_essential_genes'
]

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

for col in double_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})

print("Type conversion complete")
print(df.dtypes.value_counts())

## 5. Dataset Overview

In [ ]:
total = len(df)

print("Dataset Overview")
print("=" * 60)
print(f"Total SVs         : {total:,}")
print(f"Total columns     : {len(df.columns)}")
print(f"Unique chromosomes: {df['chromosome'].nunique()}")
print()
print("Column list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:<45} {str(df[col].dtype)}")

## 6. Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum().values,
    'missing_pct': (df.isnull().sum().values / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

missing_with_nulls = missing[missing['missing_count'] > 0]
print(f"Columns with missing values: {len(missing_with_nulls)}")
print()
print(missing_with_nulls.to_string(index=False))

missing.to_csv(METRICS_DIR / 'missing_values.csv', index=False)
print(f"\nSaved: {METRICS_DIR / 'missing_values.csv'}")

## 7. Target Variable Analysis

In [ ]:
if 'sv_classification' in df.columns:
    print("SV Classification Distribution (primary target)")
    print("=" * 55)
    sv_class_dist = df['sv_classification'].value_counts()
    print(sv_class_dist.to_string())
    print(f"\nTotal classes: {sv_class_dist.shape[0]}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sv_class_dist.sort_values().plot(kind='barh', ax=axes[0],
                                      color='steelblue', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Number of SVs', fontsize=11, fontweight='bold')
    axes[0].set_title('SV Classification Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    if 'sv_pathogenicity_risk' in df.columns:
        risk_dist = df['sv_pathogenicity_risk'].value_counts()
        risk_dist.sort_values().plot(kind='barh', ax=axes[1],
                                      color='coral', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Number of SVs', fontsize=11, fontweight='bold')
        axes[1].set_title('SV Pathogenicity Risk Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '01_sv_classification.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 01_sv_classification.png")

## 8. SV Type Analysis

In [ ]:
if 'sv_type_class' in df.columns:
    print("SV Type Class Distribution")
    svtype_dist = df['sv_type_class'].value_counts()
    print(svtype_dist.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    svtype_dist.plot(kind='bar', ax=axes[0], color='mediumpurple', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('SV Type', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
    axes[0].set_title('SV Type Class Distribution', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)

    if 'variant_type' in df.columns:
        vtype_dist = df['variant_type'].value_counts()
        vtype_dist.plot(kind='bar', ax=axes[1], color='teal', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Variant Type', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Count', fontsize=11, fontweight='bold')
        axes[1].set_title('Variant Type Distribution', fontsize=12, fontweight='bold')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_sv_types.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_sv_types.png")

## 9. SV Size Analysis

In [ ]:
if 'sv_size' in df.columns:
    print("SV Size Statistics")
    sv_size = df['sv_size'].dropna()
    print(f"Min    : {sv_size.min():,} bp")
    print(f"Median : {sv_size.median():,.0f} bp")
    print(f"Mean   : {sv_size.mean():,.0f} bp")
    print(f"Max    : {sv_size.max():,} bp")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].hist(np.log10(sv_size[sv_size > 0] + 1), bins=50,
                 color='darkorange', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('log10(SV Size in bp)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[0].set_title('SV Size Distribution (log scale)', fontsize=12, fontweight='bold')
    axes[0].grid(alpha=0.3)

    if 'sv_size_category' in df.columns:
        size_cat_dist = df['sv_size_category'].value_counts()
        size_cat_dist.sort_values().plot(kind='barh', ax=axes[1],
                                          color='darkorange', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
        axes[1].set_title('SV Size Category Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '03_sv_size.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 03_sv_size.png")

## 10. Gene Overlap Analysis

In [ ]:
if 'genes_overlapped' in df.columns:
    print("Genes Overlapped per SV")
    genes_ov = df['genes_overlapped'].dropna()
    print(f"Min    : {genes_ov.min()}")
    print(f"Median : {genes_ov.median():.1f}")
    print(f"Mean   : {genes_ov.mean():.1f}")
    print(f"Max    : {genes_ov.max()}")
    print(f"SVs with 0 genes : {(genes_ov == 0).sum():,} ({(genes_ov == 0).sum()/total*100:.1f}%)")
    print(f"SVs with 1+ genes: {(genes_ov > 0).sum():,} ({(genes_ov > 0).sum()/total*100:.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].hist(genes_ov[genes_ov <= 50], bins=50, color='steelblue', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Genes Overlapped (capped at 50)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[0].set_title('Genes Overlapped per SV', fontsize=12, fontweight='bold')
    axes[0].grid(alpha=0.3)

    if 'gene_count_category' in df.columns:
        gcat_dist = df['gene_count_category'].value_counts()
        gcat_dist.sort_values().plot(kind='barh', ax=axes[1],
                                      color='steelblue', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
        axes[1].set_title('Gene Count Category Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '04_gene_overlap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 04_gene_overlap.png")

## 11. Gene Category Impacts

In [ ]:
gene_cat_cols = [
    'pharmacogenes_affected', 'omim_genes_affected',
    'kinase_genes_affected', 'receptor_genes_affected',
    'cancer_genes_affected', 'neuro_genes_affected',
    'broadly_expressed_genes_affected'
]
gene_cat_cols = [c for c in gene_cat_cols if c in df.columns]

print("Gene Category Impact Statistics")
print("=" * 55)
for col in gene_cat_cols:
    data = df[col].dropna()
    nonzero = (data > 0).sum()
    print(f"  {col:<40} : nonzero={nonzero:,} ({nonzero/total*100:.1f}%)")

n_cols = 3
n_rows = (len(gene_cat_cols) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(gene_cat_cols):
    data = df[col].dropna()
    data_nonzero = data[data > 0]
    axes[i].hist(data_nonzero, bins=30, color='coral', alpha=0.8, edgecolor='black')
    axes[i].set_title(f'{col}\n(nonzero only)', fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Count', fontsize=8)
    axes[i].set_ylabel('Frequency', fontsize=8)
    axes[i].grid(alpha=0.3)

for i in range(len(gene_cat_cols), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '05_gene_category_impacts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 05_gene_category_impacts.png")

## 12. Clinical Priority and Impact Tier

In [ ]:
if 'sv_impact_tier' in df.columns:
    print("SV Impact Tier Distribution")
    tier_dist = df['sv_impact_tier'].value_counts()
    print(tier_dist.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    tier_dist.sort_values().plot(kind='barh', ax=axes[0],
                                  color='mediumpurple', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[0].set_title('SV Impact Tier Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    if 'sv_clinical_priority' in df.columns:
        clin_dist = df['sv_clinical_priority'].value_counts()
        clin_dist.sort_values().plot(kind='barh', ax=axes[1],
                                      color='darkorange', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
        axes[1].set_title('SV Clinical Priority Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '06_sv_impact_clinical_priority.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 06_sv_impact_clinical_priority.png")

## 13. Boolean Feature Analysis

In [ ]:
bool_features = [
    'has_critical_gene_disruption',
    'has_disease_associated_genes',
    'affects_essential_genes'
]
bool_features = [c for c in bool_features if c in df.columns]

bool_counts = {col: int(df[col].sum()) for col in bool_features}

print("Boolean Feature Summary")
print("=" * 50)
for col, count in bool_counts.items():
    print(f"  {col:<40} : {count:>8,}  ({count/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
names  = [c.replace('_', ' ').title() for c in bool_features]
values = [bool_counts[c] for c in bool_features]
colors = ['#e74c3c', '#3498db', '#f39c12']

bars = ax.bar(names, values, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val:,}\n({val/total*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Count', fontsize=11, fontweight='bold')
ax.set_title('Boolean Feature Distribution', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '07_boolean_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 07_boolean_features.png")

## 14. Gene Disruption and Druggability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if 'max_gene_disruption_fraction' in df.columns:
    data = df['max_gene_disruption_fraction'].dropna()
    axes[0].hist(data, bins=50, color='darkred', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Max Gene Disruption Fraction', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[0].set_title('Max Gene Disruption Fraction Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(alpha=0.3)
    print(f"Max Gene Disruption Fraction - median: {data.median():.3f}")

if 'avg_druggability_affected_genes' in df.columns:
    data = df['avg_druggability_affected_genes'].dropna()
    axes[1].hist(data, bins=40, color='darkgreen', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Avg Druggability of Affected Genes', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[1].set_title('Avg Druggability of Affected Genes', fontsize=12, fontweight='bold')
    axes[1].grid(alpha=0.3)
    print(f"Avg Druggability Affected Genes - median: {data.median():.3f}")

plt.tight_layout()
plt.savefig(IMAGES_DIR / '08_gene_disruption_druggability.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 08_gene_disruption_druggability.png")

## 15. SV Combined Impact Score

In [ ]:
if 'sv_combined_impact_score' in df.columns:
    print("SV Combined Impact Score Distribution")
    score_data = df['sv_combined_impact_score'].dropna()
    print(score_data.describe().to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].hist(score_data, bins=40, color='navy', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('SV Combined Impact Score', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[0].set_title('SV Combined Impact Score Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(alpha=0.3)

    if 'sv_classification' in df.columns:
        df.boxplot(column='sv_combined_impact_score', by='sv_classification',
                   ax=axes[1], patch_artist=True)
        axes[1].set_xlabel('SV Classification', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Combined Impact Score', fontsize=11, fontweight='bold')
        axes[1].set_title('Impact Score by SV Classification', fontsize=12, fontweight='bold')
        plt.suptitle('')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '09_sv_combined_impact_score.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 09_sv_combined_impact_score.png")

## 16. Correlation Analysis

In [ ]:
corr_features = [
    'sv_size', 'genes_overlapped', 'pharmacogenes_affected',
    'omim_genes_affected', 'kinase_genes_affected',
    'total_disease_associations', 'cancer_genes_affected',
    'neuro_genes_affected', 'broadly_expressed_genes_affected',
    'sv_combined_impact_score', 'max_gene_disruption_fraction',
    'avg_druggability_affected_genes'
]
corr_features = [c for c in corr_features if c in df.columns]

corr_data   = df[corr_features].apply(pd.to_numeric, errors='coerce')
corr_matrix = corr_data.corr()

corr_matrix.to_csv(METRICS_DIR / 'correlation_matrix.csv')
print(f"Saved: {METRICS_DIR / 'correlation_matrix.csv'}")

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '10_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 10_correlation_matrix.png")

print("\nHighly correlated pairs (|r| > 0.9):")
found = False
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.9:
            print(f"  {corr_matrix.columns[i]} vs {corr_matrix.columns[j]}: {val:.3f}")
            found = True
if not found:
    print("  None found above 0.9")

## 17. Feature Statistics Summary

In [ ]:
stats = df[corr_features].describe().T
stats['missing_pct'] = (df[corr_features].isnull().sum() / len(df) * 100).values
stats.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved: {METRICS_DIR / 'feature_statistics.csv'}")
print()
print(stats.to_string())

## 18. EDA Report

In [ ]:
report_path      = REPORTS_DIR / 'structural_variant_eda_report.txt'
images_generated = sorted(IMAGES_DIR.glob('*.png'))

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("STRUCTURAL VARIANT ML FEATURES - EDA REPORT\n")
    f.write("DNA Gene Mapping Project - ML Phase V5\n")
    f.write("=" * 80 + "\n\n")

    f.write("TABLE: gold.structural_variant_ml_features\n")
    f.write(f"Total rows    : {len(df):,} (full table loaded)\n")
    f.write(f"Columns       : {len(df.columns)}\n")
    f.write(f"Chromosomes   : {df['chromosome'].nunique()}\n\n")

    if 'sv_classification' in df.columns:
        f.write("TARGET VARIABLE (sv_classification)\n")
        f.write("-" * 40 + "\n")
        for cls, count in df['sv_classification'].value_counts().items():
            f.write(f"  {str(cls):<30} : {count:,} ({count/total*100:.1f}%)\n")
        f.write("\n")

    if 'sv_size' in df.columns:
        sv_size = df['sv_size'].dropna()
        f.write("SV SIZE DISTRIBUTION\n")
        f.write("-" * 40 + "\n")
        f.write(f"  Min    : {sv_size.min():,} bp\n")
        f.write(f"  Median : {sv_size.median():,.0f} bp\n")
        f.write(f"  Max    : {sv_size.max():,} bp\n\n")

    if 'genes_overlapped' in df.columns:
        genes_ov = df['genes_overlapped'].dropna()
        f.write("GENE OVERLAP\n")
        f.write("-" * 40 + "\n")
        f.write(f"  Median genes overlapped : {genes_ov.median():.1f}\n")
        f.write(f"  SVs with 0 genes        : {(genes_ov == 0).sum():,} ({(genes_ov==0).sum()/total*100:.1f}%)\n\n")

    f.write("MISSING VALUES\n")
    f.write("-" * 40 + "\n")
    f.write(f"Columns with missing data: {len(missing_with_nulls)}\n")
    if len(missing_with_nulls) > 0:
        for _, row in missing_with_nulls.head(10).iterrows():
            f.write(f"  {row['column']:<45} {row['missing_pct']:.1f}%\n")
    f.write("\n")

    f.write("VISUALIZATIONS GENERATED\n")
    f.write("-" * 40 + "\n")
    for img in images_generated:
        f.write(f"  {img.name}\n")
    f.write("\n")

    f.write("OUTPUT FILES\n")
    f.write("-" * 40 + "\n")
    f.write(f"  images/  : {len(images_generated)} PNG files\n")
    f.write(f"  reports/ : structural_variant_eda_report.txt\n")
    f.write(f"  metrics/ : missing_values.csv, correlation_matrix.csv, feature_statistics.csv\n")
    f.write("\n")

    f.write("ML TRAINING NOTES\n")
    f.write("-" * 40 + "\n")
    f.write("  1. Drop gene_list column (pipe-delimited string, not usable directly)\n")
    f.write("  2. Use genes_overlapped (INT) as the gene count feature\n")
    f.write("  3. Cross-validate due to smaller dataset (216K rows)\n")
    f.write("  4. Stratify by sv_type_class when splitting\n")
    f.write("  5. Proceed to 06_variant_tables_eda.ipynb\n")

print(f"Saved: {report_path}")
print()
print("=" * 60)
print("STRUCTURAL VARIANT ML FEATURES EDA COMPLETE")
print("=" * 60)
print(f"  Images   : {len(images_generated)}")
print(f"  Reports  : 1")
print(f"  Metrics  : 3 CSV files")
print(f"  Output   : {BASE_OUT}")
print()
print("Next: 06_variant_tables_eda.ipynb")